In [2]:
import bs4
import lxml
import pandas as pd
import urllib
from selenium.webdriver import Chrome
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By

import re
from bs4 import SoupStrainer

from urllib import request

In [3]:
url_base = "https://www.bundestag.de/parlament/praesidium/parteienfinanzierung/fundstellen50000/"
year  = "2024"

In [4]:
url_full = url_base + year
only_table = SoupStrainer('table')
table_request_text = request.urlopen(url_full).read()
table = bs4.BeautifulSoup(table_request_text, "html.parser",parse_only=only_table)
raw_table = table.find_all('table', 'table')[0]

In [5]:
df = pd.read_html(str(raw_table))[0]
df

/tmp/ipykernel_3827/1469236618.py:1: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  df = pd.read_html(str(raw_table))[0]


,Partei,Spende,Spender,Eingang der Spende oder der Spendenan-­ kündigung,"Eingang der Anzeige, Drucksache"
,Juli 2024,Juli 2024,Juli 2024,Juli 2024,Juli 2024
,April 2024,April 2024,April 2024,April 2024,April 2024
0,Dezember 2024,Dezember 2024,Dezember 2024,Dezember 2024,Dezember 2024
1,SPD,35.001 Euro,Verband der Bayerischen Metall und Elektro-Ind...,30.12.2024,30.12.2024
2,FDP,50.000 Euro,Verband der Bayerischen Metall- und Elektro-In...,30.12.2024,30.12.2024
3,FDP,100.000 Euro,Jörg Bantleon Arcisstr. 50 80799 München,27.12.2024,30.12.2024
4,FDP,200.000 Euro,Futrue GmbH Am Haag 14 82166 Gräfelfing,27.12.2024,30.12.2024
...,...,...,...,...,...
141,Bündnis 90/ Die Grünen,50.001 Euro,Verband der Bayerischen Metall- und Elektro-In...,27.12.2023,30.01.2024 Drs. 20/10781
142,FDP,51.000 Euro,Ulrich Horst Marseille Sportallee 1 22335 Hamburg,15.1.2024,16.01.2024 Drs. 20/10781


In [35]:
dfs = []
# get table
year = 2024
for year in range(2009,2026):
    url_full = url_base + str(year)
    only_table = SoupStrainer('table')
    table_request_text = request.urlopen(url_full).read()
    table = bs4.BeautifulSoup(table_request_text, "html.parser",parse_only=only_table)
    raw_table = table.find_all('table', 'table')[0]
    df = pd.read_html(str(raw_table))[0]

    # clean up
    df_dons = df.iloc[:,: 5].set_axis(['Partie', 'Montant', 'Donneur', 'Date', 'Date Publication'], axis=1)
    df_dons = df_dons[df_dons['Montant'].str.match('\d+\.\d\d\d')]
    df_dons = df_dons[df_dons['Date'].str.match('\d{1,2}.\d\d.\d\d\d\d$', flags=re.U)]
    df_dons['Montant'] = df_dons['Montant'].apply(lambda x: re.search(r'\d{1,3}(\.\d{3})*', x).group(0))
    df_dons['Montant'] = df_dons['Montant'].apply(lambda x: int(x.replace('.','')))
    df_dons['Date'] = pd.to_datetime(df_dons['Date'],format= '%d.%m.%Y')
    df_dons
    dfs.append(df_dons)

df_don_complet = pd.concat(dfs)
df_don_complet

/tmp/ipykernel_3827/2884208799.py:10: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  df = pd.read_html(str(raw_table))[0]
/tmp/ipykernel_3827/2884208799.py:10: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  df = pd.read_html(str(raw_table))[0]
/tmp/ipykernel_3827/2884208799.py:10: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  df = pd.read_html(str(raw_table))[0]
/tmp/ipykernel_3827/2884208799.py:10: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  df = pd.read_html(str(raw_table))[0]
/tmp/ipykernel_3

,Partie,Montant,Donneur,Date,Date Publication
1,CDU,100000,Südwestmetall Verband der Metall- und Elektroi...,2009-12-28,29.12.2009
5,CDU,150000,Frau Johanna Quandt Seedammweg 55 61352 Bad Ho...,2009-10-01,02.10.2009
6,CDU,150000,Herr Stefan Quandt Seedammweg 55 61352 Bad Hom...,2009-10-01,02.10.2009
7,CDU,150000,Frau Susanne Klatten Seedammweg 55 61352 Bad H...,2009-10-01,02.10.2009
8,FDP1,300000,Substantia AG Berliner Allee 21 40212 Düsseldorf,2009-10-13,19.10.2009
...,...,...,...,...,...
68,FDP,50000,Johannes Peter Huth 61 Rutland Gate London SW7...,2025-01-02,03.01.2025
69,CDU,50000,Johannes Peter Huth 61 Rutland Gate London SW7...,2025-01-02,03.01.2025
70,Bündnis 90/ Die Grünen,35001,vbm- Verband der Bayerischen Metall- und Elekt...,2024-12-30,02.01.2025
71,CDU,167000,Torsten Toeller c/o Allegro Invest SE Westpreu...,2024-12-27,02.01.2025
